# 06 - RAG Chunking and Vector Storage Deep Dive

This notebook demonstrates the **complete RAG (Retrieval-Augmented Generation) pipeline** from document chunking to vector storage and semantic search. You'll learn how text gets processed, embedded, and stored for retrieval.

## 🧠 What You'll Learn
- **Document Chunking**: How to split large texts into searchable segments
- **Embedding Creation**: Converting text chunks to vector representations
- **Vector Storage**: Storing embeddings with metadata for retrieval
- **Similarity Search**: Finding relevant chunks using cosine similarity
- **RAG Integration**: Combining retrieval with generation

## 🎯 Learning Objectives
- Understand the complete RAG pipeline step-by-step
- See how embeddings enable semantic search
- Learn chunking strategies for different document types
- Practice vector similarity calculations
- Build a mini RAG system from scratch

## 📊 RAG Pipeline Overview

```
📄 Document → 🔪 Chunking → 🧠 Embedding → 🗃️ Vector Store → 🔍 Search → 🤖 Generation
```

| Step | Input | Process | Output |
|------|-------|---------|--------|
| **Chunking** | Large text | Split by sentences/tokens | Text segments |
| **Embedding** | Text chunks | OpenAI text-embedding-3-large | 3072-dim vectors |
| **Storage** | Vectors + metadata | Vector database | Searchable index |
| **Retrieval** | User query | Similarity search | Relevant chunks |
| **Generation** | Query + context | LLM completion | Final answer |

In [1]:
import os
import json
import numpy as np
from openai import AzureOpenAI
from dotenv import load_dotenv
import re
from typing import List, Dict, Tuple
import time
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()

# Azure OpenAI Configuration
client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-05-01-preview"
)

# Model deployments
gpt_deployment = os.getenv("AZURE_OPENAI_GPT4O_DEPLOYMENT_NAME")
embeddings_deployment = os.getenv("AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT_NAME")

print("✅ Azure OpenAI client initialized for RAG pipeline")
print(f"🤖 GPT Model: {gpt_deployment}")
print(f"🧠 Embeddings Model: {embeddings_deployment}")
print(f"📐 Expected embedding dimensions: 3072 (text-embedding-3-large)")

✅ Azure OpenAI client initialized for RAG pipeline
🤖 GPT Model: gpt-4o
🧠 Embeddings Model: text-embedding-3-large
📐 Expected embedding dimensions: 3072 (text-embedding-3-large)


## 📚 Understanding Libraries and Embedding Dimensions

Before we dive into the RAG pipeline, let's understand the key libraries we're using and what embedding dimensions mean.

### 📦 **New Libraries in This Notebook**

**NumPy (`numpy`)**: 
- Essential for numerical computing and vector operations
- Used for storing and manipulating embedding vectors
- Enables efficient similarity calculations

**Scikit-learn (`sklearn`)**:
- Machine learning library providing similarity metrics
- We use `cosine_similarity` to find similar vectors
- Much more efficient than manual similarity calculations

### 📐 **What Are Embedding Dimensions?**

**Embedding dimensions** are the length of the vector that represents your text:

- **text-embedding-3-large**: 3072 dimensions (our model)
- **text-embedding-3-small**: 1536 dimensions (smaller, faster)
- **text-embedding-ada-002**: 1536 dimensions (older model)

**Think of dimensions like:**
- 📊 **Coordinates in space**: Each dimension is like X, Y, Z coordinates but with 3072 axes
- 🎯 **Features of meaning**: Each dimension captures different aspects of semantic meaning
- 📏 **Vector length**: More dimensions = more nuanced understanding of text meaning

**Why 3072 dimensions?**
- More dimensions can capture more nuanced semantic relationships
- But also require more storage and computation
- OpenAI chose 3072 as optimal balance of quality vs efficiency

**🔑 CRITICAL INSIGHT: Fixed Dimensions Regardless of Text Length**

```python
# ALL of these texts become 3072-dimensional vectors:
short_text = "AI"
medium_text = "Azure OpenAI is powerful for building applications"
long_text = """Azure OpenAI Service provides REST API access to OpenAI's 
powerful language models including GPT-4, GPT-3.5-turbo, and embeddings model 
series. These models can be easily adapted to your specific task including but 
not limited to content generation, summarization, semantic search, and natural 
language to code translation."""

# After embedding:
embedding_short = [0.1, -0.3, 0.7, ..., 0.2]    # 3072 numbers
embedding_medium = [0.4, -0.1, 0.9, ..., -0.1]  # 3072 numbers  
embedding_long = [0.2, -0.8, 0.3, ..., 0.5]     # 3072 numbers

len(embedding_short)  # Returns: 3072
len(embedding_medium) # Returns: 3072
len(embedding_long)   # Returns: 3072
```

**Why Fixed Dimensions?**
- 🎯 **Standardization**: All vectors must be same length for similarity calculations
- 🧠 **Model Architecture**: The neural network produces fixed-size outputs
- 📊 **Efficiency**: Consistent dimensions enable optimized mathematical operations
- 🔍 **Comparison**: You can only compare vectors of the same dimensionality

**How Does This Work?**
- **Short text**: Model distributes meaning across all 3072 dimensions
- **Long text**: Model compresses/summarizes meaning into same 3072 dimensions
- **Information density**: Longer texts may lose some detail, shorter texts may have sparse representations
- **Semantic essence**: The model captures the core meaning regardless of length

**Key Insight**: Two texts with similar meanings will have similar embedding vectors, regardless of their length!

## 📄 Step 1: Create Sample Documents

Let's create some realistic documents that we'll chunk and store in our vector database.

In [2]:
def create_sample_documents() -> Dict[str, str]:
    """Create sample documents for chunking demonstration"""
    
    documents = {
        "azure_openai_performance": """
Azure OpenAI Performance Best Practices: Use connection pooling to reduce latency. 
Implement exponential backoff for rate limiting. Cache responses when possible. 
Monitor token usage to optimize costs. Use streaming for real-time applications. 
Consider model selection based on your use case - GPT-4o for complex reasoning, 
GPT-3.5 for faster responses. Enable content filtering to prevent harmful content. 
Use private endpoints for network isolation. Implement proper RBAC controls.
        """.strip(),
        
        "azure_openai_security": """
Azure OpenAI Security Best Practices: Use Azure AD authentication and managed identities. 
Implement proper RBAC controls. Enable content filtering to prevent harmful content. 
Use private endpoints for network isolation. Regularly rotate API keys. 
Monitor access logs for suspicious activity. Implement input validation and sanitization. 
Use Azure Key Vault for secret management. Enable diagnostic logging for audit trails.
Set up alert rules for unusual usage patterns.
        """.strip(),
        
        "azure_openai_cost": """
Azure OpenAI Cost Management: Monitor token usage with Azure Monitor. 
Set up billing alerts and quotas. Use prompt engineering to reduce token consumption. 
Choose appropriate models for your use case. Implement caching strategies. 
Use batch processing when possible. Consider using smaller models for simple tasks.
Optimize prompts to reduce input and output tokens. Use streaming to improve perceived performance.
Implement rate limiting to control costs. Monitor costs across different deployments.
        """.strip(),
        
        "azure_openai_troubleshooting": """
Azure OpenAI Troubleshooting Guide: Check regional proximity to deployment for latency issues. 
Review concurrent request limits if experiencing throttling. Implement exponential backoff for rate limiting. 
Monitor requests per minute (RPM) limits. Consider provisioned throughput for predictable workloads.
Verify API key and endpoint configuration. Ensure deployment name matches configuration.
Check that API version supports required features. Monitor content filtering logs for blocked content.
Use Application Insights for detailed telemetry. Set up alerts for quota limits.
        """.strip()
    }
    
    print("📄 Sample documents created:")
    for doc_id, content in documents.items():
        print(f"   📋 {doc_id}: {len(content)} characters")
    
    return documents

# Create our knowledge base
documents = create_sample_documents()

📄 Sample documents created:
   📋 azure_openai_performance: 488 characters
   📋 azure_openai_security: 474 characters
   📋 azure_openai_cost: 503 characters
   📋 azure_openai_troubleshooting: 580 characters


## 🔪 Step 2: Document Chunking Strategies

Let's implement different chunking strategies to see how they affect retrieval quality.

In [3]:
class DocumentChunker:
    """Implements various document chunking strategies"""
    
    def __init__(self, chunk_size: int = 500, overlap: int = 50):
        self.chunk_size = chunk_size
        self.overlap = overlap
    
    def sentence_based_chunking(self, text: str) -> List[str]:
        """Split text by sentences with configurable grouping"""
        # Split by sentences (basic approach)
        sentences = re.split(r'[.!?]+', text)
        sentences = [s.strip() for s in sentences if s.strip()]
        
        chunks = []
        current_chunk = []
        current_length = 0
        
        for sentence in sentences:
            sentence_length = len(sentence)
            
            # If adding this sentence exceeds chunk size, finalize current chunk
            if current_length + sentence_length > self.chunk_size and current_chunk:
                chunks.append('. '.join(current_chunk) + '.')
                
                # Start new chunk with overlap
                overlap_sentences = current_chunk[-1:] if self.overlap > 0 else []
                current_chunk = overlap_sentences + [sentence]
                current_length = sum(len(s) for s in current_chunk)
            else:
                current_chunk.append(sentence)
                current_length += sentence_length
        
        # Add final chunk
        if current_chunk:
            chunks.append('. '.join(current_chunk) + '.')
        
        return chunks
    
    def token_based_chunking(self, text: str, tokens_per_chunk: int = 150) -> List[str]:
        """Split text by approximate token count (using word approximation)"""
        # Rough approximation: 1 token ≈ 0.75 words
        words_per_chunk = int(tokens_per_chunk * 0.75)
        words = text.split()
        
        chunks = []
        for i in range(0, len(words), words_per_chunk):
            chunk_words = words[i:i + words_per_chunk]
            chunks.append(' '.join(chunk_words))
        
        return chunks
    
    def semantic_chunking(self, text: str) -> List[str]:
        """Split text by semantic boundaries (topic changes)"""
        # Simple implementation: split by paragraph-like structures
        # In production, you might use more sophisticated NLP
        paragraphs = text.split('\n\n')
        chunks = []
        
        for paragraph in paragraphs:
            if len(paragraph.strip()) > 0:
                # If paragraph is too long, use sentence chunking
                if len(paragraph) > self.chunk_size:
                    sub_chunks = self.sentence_based_chunking(paragraph)
                    chunks.extend(sub_chunks)
                else:
                    chunks.append(paragraph.strip())
        
        return chunks

def demonstrate_chunking_strategies():
    """Show different chunking approaches on sample text"""
    
    chunker = DocumentChunker(chunk_size=200, overlap=30)
    sample_text = documents["azure_openai_performance"]
    
    print("🔪 CHUNKING STRATEGY COMPARISON")
    print("=" * 60)
    print(f"📄 Original text length: {len(sample_text)} characters")
    print(f"📝 Sample text: {sample_text[:100]}...")
    
    # Sentence-based chunking
    sentence_chunks = chunker.sentence_based_chunking(sample_text)
    print(f"\n📋 Sentence-based chunking: {len(sentence_chunks)} chunks")
    for i, chunk in enumerate(sentence_chunks, 1):
        print(f"   Chunk {i}: {chunk[:50]}... ({len(chunk)} chars)")
    
    # Token-based chunking
    token_chunks = chunker.token_based_chunking(sample_text, tokens_per_chunk=50)
    print(f"\n🎯 Token-based chunking: {len(token_chunks)} chunks")
    for i, chunk in enumerate(token_chunks, 1):
        print(f"   Chunk {i}: {chunk[:50]}... ({len(chunk)} chars)")
    
    return sentence_chunks

# Demonstrate chunking
demo_chunks = demonstrate_chunking_strategies()

🔪 CHUNKING STRATEGY COMPARISON
📄 Original text length: 488 characters
📝 Sample text: Azure OpenAI Performance Best Practices: Use connection pooling to reduce latency. 
Implement expone...

📋 Sentence-based chunking: 3 chunks
   Chunk 1: Azure OpenAI Performance Best Practices: Use conne... (201 chars)
   Chunk 2: Monitor token usage to optimize costs. Use streami... (192 chars)
   Chunk 3: 5 for faster responses. Enable content filtering t... (153 chars)

🎯 Token-based chunking: 2 chunks
   Chunk 1: Azure OpenAI Performance Best Practices: Use conne... (277 chars)
   Chunk 2: your use case - GPT-4o for complex reasoning, GPT-... (205 chars)


## 🧠 Step 3: Generate Embeddings for Chunks

Now let's convert our text chunks into vector embeddings using Azure OpenAI.

In [4]:
class EmbeddingGenerator:
    """Handles embedding generation with batching and error handling"""
    
    def __init__(self, client: AzureOpenAI, deployment_name: str):
        self.client = client
        self.deployment_name = deployment_name
        self.embedding_cache = {}  # Simple cache to avoid re-embedding
    
    def generate_embedding(self, text: str) -> List[float]:
        """Generate embedding for a single text"""
        
        # Check cache first
        if text in self.embedding_cache:
            return self.embedding_cache[text]
        
        try:
            response = self.client.embeddings.create(
                model=self.deployment_name,
                input=text.replace('\n', ' ').strip()
            )
            
            embedding = response.data[0].embedding
            self.embedding_cache[text] = embedding
            
            return embedding
            
        except Exception as e:
            print(f"❌ Error generating embedding: {e}")
            return None
    
    def generate_batch_embeddings(self, texts: List[str], batch_size: int = 10) -> List[Tuple[str, List[float]]]:
        """Generate embeddings for multiple texts with batching"""
        
        results = []
        
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            print(f"🧠 Processing batch {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1} ({len(batch)} texts)")
            
            for text in batch:
                embedding = self.generate_embedding(text)
                if embedding:
                    results.append((text, embedding))
                    print(f"   ✅ Generated embedding: {len(embedding)} dimensions")
                else:
                    print(f"   ❌ Failed to generate embedding for text")
            
            # Rate limiting - be nice to the API
            if i + batch_size < len(texts):
                time.sleep(1)
        
        return results

def process_documents_to_embeddings():
    """Complete pipeline: documents → chunks → embeddings"""
    
    print("🧠 EMBEDDING GENERATION PIPELINE")
    print("=" * 60)
    
    # Initialize components
    chunker = DocumentChunker(chunk_size=300, overlap=50)
    embedder = EmbeddingGenerator(client, embeddings_deployment)
    
    # Process all documents
    all_chunks = []
    
    for doc_id, content in documents.items():
        print(f"\n📄 Processing document: {doc_id}")
        
        # Chunk the document
        chunks = chunker.sentence_based_chunking(content)
        print(f"   🔪 Generated {len(chunks)} chunks")
        
        # Add metadata to chunks
        for i, chunk in enumerate(chunks):
            chunk_data = {
                "text": chunk,
                "document_id": doc_id,
                "chunk_index": i,
                "char_count": len(chunk),
                "source": "azure_openai_guide"
            }
            all_chunks.append(chunk_data)
    
    print(f"\n📊 Total chunks to embed: {len(all_chunks)}")
    
    # Generate embeddings
    print(f"\n🧠 Generating embeddings...")
    texts_to_embed = [chunk["text"] for chunk in all_chunks]
    embeddings_results = embedder.generate_batch_embeddings(texts_to_embed)
    
    # Combine chunks with embeddings
    embedded_chunks = []
    for i, (text, embedding) in enumerate(embeddings_results):
        chunk_data = all_chunks[i]
        chunk_data["embedding"] = embedding
        chunk_data["embedding_dims"] = len(embedding)
        embedded_chunks.append(chunk_data)
    
    print(f"\n✅ Embedding generation completed!")
    print(f"   📊 Embedded chunks: {len(embedded_chunks)}")
    print(f"   📐 Embedding dimensions: {embedded_chunks[0]['embedding_dims'] if embedded_chunks else 'N/A'}")
    
    return embedded_chunks

# Generate embeddings for all chunks
embedded_chunks = process_documents_to_embeddings()

🧠 EMBEDDING GENERATION PIPELINE

📄 Processing document: azure_openai_performance
   🔪 Generated 2 chunks

📄 Processing document: azure_openai_security
   🔪 Generated 2 chunks

📄 Processing document: azure_openai_cost
   🔪 Generated 2 chunks

📄 Processing document: azure_openai_troubleshooting
   🔪 Generated 3 chunks

📊 Total chunks to embed: 9

🧠 Generating embeddings...
🧠 Processing batch 1/1 (9 texts)
   ✅ Generated embedding: 3072 dimensions
   ✅ Generated embedding: 3072 dimensions
   ✅ Generated embedding: 3072 dimensions
   ✅ Generated embedding: 3072 dimensions
   ✅ Generated embedding: 3072 dimensions
   ✅ Generated embedding: 3072 dimensions
   ✅ Generated embedding: 3072 dimensions
   ✅ Generated embedding: 3072 dimensions
   ✅ Generated embedding: 3072 dimensions

✅ Embedding generation completed!
   📊 Embedded chunks: 9
   📐 Embedding dimensions: 3072


## 🗃️ Step 4: Create Vector Store and Storage

Let's implement a simple in-memory vector store to understand how vector databases work.

In [5]:
class SimpleVectorStore:
    """Simple in-memory vector store for demonstration"""
    
    def __init__(self):
        self.vectors = []
        self.metadata = []
        self.index_map = {}  # chunk_id -> index mapping
    
    def add_vectors(self, embedded_chunks: List[Dict]):
        """Add embedded chunks to the vector store"""
        
        print("🗃️ Adding vectors to store...")
        
        for chunk_data in embedded_chunks:
            if chunk_data.get("embedding"):
                # Generate unique ID
                chunk_id = f"{chunk_data['document_id']}_chunk_{chunk_data['chunk_index']}"
                
                # Store vector and metadata
                vector_idx = len(self.vectors)
                self.vectors.append(np.array(chunk_data["embedding"]))
                
                metadata = {
                    "chunk_id": chunk_id,
                    "text": chunk_data["text"],
                    "document_id": chunk_data["document_id"],
                    "chunk_index": chunk_data["chunk_index"],
                    "char_count": chunk_data["char_count"],
                    "source": chunk_data.get("source", "unknown")
                }
                self.metadata.append(metadata)
                self.index_map[chunk_id] = vector_idx
                
                print(f"   ✅ Added: {chunk_id} ({len(chunk_data['embedding'])} dims)")
        
        print(f"\n📊 Vector store status:")
        print(f"   📋 Total vectors: {len(self.vectors)}")
        print(f"   📐 Vector dimensions: {len(self.vectors[0]) if self.vectors else 0}")
        print(f"   🗂️ Metadata entries: {len(self.metadata)}")
    
    def similarity_search(self, query_embedding: List[float], top_k: int = 3) -> List[Dict]:
        """Find most similar vectors using cosine similarity"""
        
        if not self.vectors:
            return []
        
        # Convert query to numpy array
        query_vector = np.array(query_embedding).reshape(1, -1)
        
        # Convert stored vectors to matrix
        vector_matrix = np.vstack(self.vectors)
        
        # Calculate cosine similarities
        similarities = cosine_similarity(query_vector, vector_matrix)[0]
        
        # Get top-k most similar
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            result = {
                "similarity_score": float(similarities[idx]),
                "metadata": self.metadata[idx],
                "text": self.metadata[idx]["text"]
            }
            results.append(result)
        
        return results
    
    def get_stats(self) -> Dict:
        """Get vector store statistics"""
        if not self.vectors:
            return {"count": 0}
        
        similarities_matrix = cosine_similarity(np.vstack(self.vectors))
        np.fill_diagonal(similarities_matrix, 0)  # Remove self-similarity
        
        return {
            "total_vectors": len(self.vectors),
            "dimensions": len(self.vectors[0]),
            "avg_similarity": float(np.mean(similarities_matrix)),
            "max_similarity": float(np.max(similarities_matrix)),
            "min_similarity": float(np.min(similarities_matrix))
        }

# Create and populate vector store
vector_store = SimpleVectorStore()
vector_store.add_vectors(embedded_chunks)

# Show vector store statistics
stats = vector_store.get_stats()
print(f"\n📊 VECTOR STORE STATISTICS")
print("=" * 40)
for key, value in stats.items():
    print(f"   {key}: {value}")

🗃️ Adding vectors to store...
   ✅ Added: azure_openai_performance_chunk_0 (3072 dims)
   ✅ Added: azure_openai_performance_chunk_1 (3072 dims)
   ✅ Added: azure_openai_security_chunk_0 (3072 dims)
   ✅ Added: azure_openai_security_chunk_1 (3072 dims)
   ✅ Added: azure_openai_cost_chunk_0 (3072 dims)
   ✅ Added: azure_openai_cost_chunk_1 (3072 dims)
   ✅ Added: azure_openai_troubleshooting_chunk_0 (3072 dims)
   ✅ Added: azure_openai_troubleshooting_chunk_1 (3072 dims)
   ✅ Added: azure_openai_troubleshooting_chunk_2 (3072 dims)

📊 Vector store status:
   📋 Total vectors: 9
   📐 Vector dimensions: 3072
   🗂️ Metadata entries: 9

📊 VECTOR STORE STATISTICS
   total_vectors: 9
   dimensions: 3072
   avg_similarity: 0.474266313728391
   max_similarity: 0.7697525260854744
   min_similarity: 0.0


## 🔍 Step 4.5: Examining Vector Store Contents

Let's look inside our vector store to understand what gets stored and how the data is organized.

In [ ]:
def examine_vector_store(vector_store: SimpleVectorStore):
    """Examine and display the contents of our vector store"""
    
    print("🔍 VECTOR STORE CONTENTS EXAMINATION")
    print("=" * 60)
    
    if not vector_store.vectors:
        print("❌ Vector store is empty!")
        return
    
    # 1. Show overall structure
    print("📊 OVERALL STRUCTURE:")
    print(f"   📋 Total vectors stored: {len(vector_store.vectors)}")
    print(f"   📐 Vector dimensions: {len(vector_store.vectors[0])}")
    print(f"   🗂️ Metadata entries: {len(vector_store.metadata)}")
    print(f"   🔗 Index mappings: {len(vector_store.index_map)}")
    
    # 2. Show sample vectors (first few values only)
    print(f"\n🧠 SAMPLE EMBEDDING VECTORS:")
    print("   (Showing first 10 dimensions of each vector)")
    print("-" * 50)
    
    for i, vector in enumerate(vector_store.vectors[:3]):  # Show first 3 vectors
        metadata = vector_store.metadata[i]
        vector_preview = vector[:10]  # First 10 dimensions
        
        print(f"\n   📍 Vector {i+1} ({metadata['chunk_id']}):")
        print(f"   📝 Text: {metadata['text'][:60]}...")
        print(f"   📊 Vector: [{', '.join([f'{v:.4f}' for v in vector_preview])}, ...]")
        print(f"   📏 Full length: {len(vector)} dimensions")
    
    # 3. Show all metadata entries
    print(f"\n🗂️ ALL METADATA ENTRIES:")
    print("-" * 50)
    
    for i, metadata in enumerate(vector_store.metadata):
        print(f"\n   📋 Entry {i+1}:")
        print(f"   🆔 Chunk ID: {metadata['chunk_id']}")
        print(f"   📄 Document: {metadata['document_id']}")
        print(f"   📍 Chunk Index: {metadata['chunk_index']}")
        print(f"   📏 Characters: {metadata['char_count']}")
        print(f"   📝 Text Preview: {metadata['text'][:80]}...")
        print(f"   🏷️ Source: {metadata['source']}")
    
    # 4. Show vector statistics
    print(f"\n📈 VECTOR STATISTICS:")
    print("-" * 30)
    
    # Convert vectors to numpy array for analysis
    vector_matrix = np.vstack(vector_store.vectors)
    
    print(f"   📊 Shape: {vector_matrix.shape}")
    print(f"   📈 Mean value: {np.mean(vector_matrix):.6f}")
    print(f"   📉 Min value: {np.min(vector_matrix):.6f}")
    print(f"   📈 Max value: {np.max(vector_matrix):.6f}")
    print(f"   📊 Standard deviation: {np.std(vector_matrix):.6f}")
    
    # 5. Show similarity matrix (how similar chunks are to each other)
    print(f"\n🔗 CHUNK SIMILARITY MATRIX:")
    print("   (How similar each chunk is to every other chunk)")
    print("-" * 50)
    
    similarity_matrix = cosine_similarity(vector_matrix)
    
    print("   📊 Similarity scores (1.0 = identical, 0.0 = unrelated):")
    
    # Create a readable similarity table
    chunk_ids = [meta['chunk_id'][:20] + "..." if len(meta['chunk_id']) > 20 else meta['chunk_id'] 
                 for meta in vector_store.metadata]
    
    # Print header
    print("   " + "│".join([f"{id:>12}" for id in chunk_ids[:4]]))  # Show first 4 for readability
    print("   " + "─" * (13 * min(4, len(chunk_ids))))
    
    # Print similarity scores
    for i in range(min(4, len(similarity_matrix))):  # Show first 4 rows
        row_scores = [f"{similarity_matrix[i][j]:>12.4f}" for j in range(min(4, len(similarity_matrix[i])))]
        print("   " + "│".join(row_scores))
    
    # 6. Show what makes vectors similar/different
    print(f"\n🎯 UNDERSTANDING SIMILARITY:")
    print("-" * 40)
    
    # Find most and least similar pair (excluding self-similarity)
    # Create a copy to avoid modifying the original matrix
    similarity_analysis = similarity_matrix.copy()
    
    # Instead of setting diagonal to 0, we'll mask it out properly
    # Create a mask for non-diagonal elements
    mask = ~np.eye(similarity_analysis.shape[0], dtype=bool)
    
    # Get upper triangle indices (excluding diagonal) to avoid duplicate pairs
    upper_triangle = np.triu(similarity_analysis, k=1)  # k=1 excludes diagonal
    
    # Find max and min in upper triangle only
    max_sim_idx = np.unravel_index(np.argmax(upper_triangle), upper_triangle.shape)
    
    # For minimum, we need to handle the case where upper_triangle has zeros
    # Only consider non-zero values in upper triangle
    upper_triangle_nonzero = upper_triangle[upper_triangle > 0]
    if len(upper_triangle_nonzero) > 0:
        min_similarity_value = np.min(upper_triangle_nonzero)
        min_sim_idx = np.where(upper_triangle == min_similarity_value)
        min_sim_idx = (min_sim_idx[0][0], min_sim_idx[1][0])  # Take first occurrence
    else:
        # Fallback: use the minimum from the full masked matrix
        masked_similarities = similarity_analysis[mask]
        min_similarity_value = np.min(masked_similarities)
        temp_matrix = similarity_analysis.copy()
        temp_matrix[~mask] = 1.0  # Set diagonal to 1.0 to exclude from min search
        min_sim_idx = np.unravel_index(np.argmin(temp_matrix), temp_matrix.shape)
    
    max_similarity = similarity_analysis[max_sim_idx]
    min_similarity = min_similarity_value if 'min_similarity_value' in locals() else similarity_analysis[min_sim_idx]
    
    print(f"   🏆 Most similar chunks (score: {max_similarity:.4f}):")
    print(f"      1️⃣ {vector_store.metadata[max_sim_idx[0]]['text'][:50]}...")
    print(f"      2️⃣ {vector_store.metadata[max_sim_idx[1]]['text'][:50]}...")
    
    print(f"\n   🎭 Least similar chunks (score: {min_similarity:.4f}):")
    print(f"      1️⃣ {vector_store.metadata[min_sim_idx[0]]['text'][:50]}...")
    print(f"      2️⃣ {vector_store.metadata[min_sim_idx[1]]['text'][:50]}...")
    
    # Add explanation of what we're seeing
    if max_similarity > 0.8:
        print(f"      💡 High similarity suggests these chunks are about related topics!")
    if min_similarity < 0.3:
        print(f"      💡 Low similarity suggests these chunks cover different topics.")
    
    # 7. Index mapping
    print(f"\n🗺️ INDEX MAPPING:")
    print("   (How chunk IDs map to vector positions)")
    print("-" * 50)
    
    for chunk_id, vector_idx in list(vector_store.index_map.items())[:5]:  # Show first 5
        print(f"   📌 '{chunk_id}' → Vector position {vector_idx}")
    
    if len(vector_store.index_map) > 5:
        print(f"   ... and {len(vector_store.index_map) - 5} more mappings")

# Examine our vector store in detail
examine_vector_store(vector_store)

🔍 VECTOR STORE CONTENTS EXAMINATION
📊 OVERALL STRUCTURE:
   📋 Total vectors stored: 9
   📐 Vector dimensions: 3072
   🗂️ Metadata entries: 9
   🔗 Index mappings: 9

🧠 SAMPLE EMBEDDING VECTORS:
   (Showing first 10 dimensions of each vector)
--------------------------------------------------

   📍 Vector 1 (azure_openai_performance_chunk_0):
   📝 Text: Azure OpenAI Performance Best Practices: Use connection pool...
   📊 Vector: [-0.0277, -0.0005, -0.0112, 0.0051, -0.0110, 0.0024, -0.0347, 0.0355, -0.0300, 0.0330, ...]
   📏 Full length: 3072 dimensions

   📍 Vector 2 (azure_openai_performance_chunk_1):
   📝 Text: Use streaming for real-time applications. Consider model sel...
   📊 Vector: [-0.0358, -0.0420, -0.0177, 0.0143, 0.0207, 0.0052, -0.0341, 0.0385, -0.0048, 0.0421, ...]
   📏 Full length: 3072 dimensions

   📍 Vector 3 (azure_openai_security_chunk_0):
   📝 Text: Azure OpenAI Security Best Practices: Use Azure AD authentic...
   📊 Vector: [-0.0100, -0.0250, -0.0034, 0.0023, 0.0152,

## 🧠 Understanding What We Just Saw

### 📊 **Vector Store Structure**
What we examined shows the core components of any vector database:

1. **📋 Vectors**: The actual numerical embeddings (3072 numbers per chunk)
2. **🗂️ Metadata**: Information about each chunk (source, ID, text, etc.)
3. **🗺️ Index Mapping**: Fast lookup from chunk ID to vector position

### 🔢 **Those Numbers in the Vectors**
The embedding vectors contain numbers like `[0.0234, -0.1456, 0.0891, ...]`:

- **Positive/Negative**: Both are normal and meaningful
- **Small Values**: Usually between -1 and 1, representing feature strengths
- **3072 Numbers**: Each captures different semantic aspects of the text
- **Not Human Readable**: You can't interpret individual numbers

### 🎯 **Similarity Scores Explained**
The similarity matrix shows how related chunks are:

- **1.0 = Identical**: Exactly the same meaning
- **0.8-0.9 = Very Similar**: Same topic, different wording
- **0.6-0.8 = Related**: Connected topics
- **0.3-0.6 = Somewhat Related**: Loose connections
- **0.0-0.3 = Unrelated**: Different topics entirely

### 💡 **Key Insights**

**Why This Matters:**
- Chunks about similar topics (like "performance" and "latency") have higher similarity scores
- Our search will find the most relevant chunks based on these similarity calculations
- The metadata helps us track where information came from for citations

**What Makes Good Embeddings:**
- ✅ Related concepts cluster together (high similarity)
- ✅ Unrelated concepts are far apart (low similarity)  
- ✅ Semantic meaning is preserved across different wordings

**Production Considerations:**
- Real vector databases optimize storage and search speed
- They use approximate search algorithms for millions of vectors
- Advanced systems use multiple embedding models for different types of content

## 🔍 Step 5: Implement Semantic Search

Now let's test our vector store with semantic search queries.

In [7]:
class SemanticSearcher:
    """Handles semantic search queries against vector store"""
    
    def __init__(self, vector_store: SimpleVectorStore, embedder: EmbeddingGenerator):
        self.vector_store = vector_store
        self.embedder = embedder
    
    def search(self, query: str, top_k: int = 3) -> List[Dict]:
        """Perform semantic search for a query"""
        
        print(f"🔍 Searching for: '{query}'")
        print("-" * 50)
        
        # Generate embedding for query
        query_embedding = self.embedder.generate_embedding(query)
        if not query_embedding:
            print("❌ Failed to generate query embedding")
            return []
        
        # Perform similarity search
        results = self.vector_store.similarity_search(query_embedding, top_k)
        
        # Display results
        print(f"📊 Found {len(results)} results:")
        for i, result in enumerate(results, 1):
            score = result["similarity_score"]
            metadata = result["metadata"]
            text_preview = result["text"][:100] + "..." if len(result["text"]) > 100 else result["text"]
            
            print(f"\n   🏆 Result {i} (similarity: {score:.4f})")
            print(f"   📄 Document: {metadata['document_id']}")
            print(f"   📝 Text: {text_preview}")
        
        return results
    
    def compare_queries(self, queries: List[str]) -> Dict[str, List[Dict]]:
        """Compare search results for multiple queries"""
        
        print("🔍 SEMANTIC SEARCH COMPARISON")
        print("=" * 60)
        
        all_results = {}
        
        for query in queries:
            print(f"\n{'='*20} QUERY: {query} {'='*20}")
            results = self.search(query, top_k=3)
            all_results[query] = results
        
        return all_results

# Create semantic searcher
searcher = SemanticSearcher(vector_store, EmbeddingGenerator(client, embeddings_deployment))

# Test with various queries
test_queries = [
    "How can I reduce latency in Azure OpenAI?",
    "What are the security best practices?",
    "How do I manage costs effectively?",
    "My API calls are getting throttled, what should I do?",
    "What should I monitor for performance issues?"
]

search_results = searcher.compare_queries(test_queries)

🔍 SEMANTIC SEARCH COMPARISON

==================== QUERY: How can I reduce latency in Azure OpenAI? ====================
🔍 Searching for: 'How can I reduce latency in Azure OpenAI?'
--------------------------------------------------
📊 Found 3 results:

   🏆 Result 1 (similarity: 0.6618)
   📄 Document: azure_openai_performance
   📝 Text: Azure OpenAI Performance Best Practices: Use connection pooling to reduce latency. Implement exponen...

   🏆 Result 2 (similarity: 0.6201)
   📄 Document: azure_openai_troubleshooting
   📝 Text: Azure OpenAI Troubleshooting Guide: Check regional proximity to deployment for latency issues. Revie...

   🏆 Result 3 (similarity: 0.5340)
   📄 Document: azure_openai_security
   📝 Text: Azure OpenAI Security Best Practices: Use Azure AD authentication and managed identities. Implement ...

==================== QUERY: What are the security best practices? ====================
🔍 Searching for: 'What are the security best practices?'
-----------------------------

## 🤖 Step 6: Complete RAG Implementation

Now let's combine retrieval with generation to create a complete RAG system.

In [8]:
class RAGSystem:
    """Complete RAG implementation with retrieval and generation"""
    
    def __init__(self, searcher: SemanticSearcher, client: AzureOpenAI, gpt_deployment: str):
        self.searcher = searcher
        self.client = client
        self.gpt_deployment = gpt_deployment
    
    def generate_answer(self, query: str, top_k: int = 3, include_citations: bool = True) -> Dict:
        """Generate answer using RAG pattern"""
        
        print(f"🤖 RAG SYSTEM: Answering '{query}'")
        print("=" * 60)
        
        # Step 1: Retrieve relevant documents
        print("🔍 Step 1: Retrieving relevant documents...")
        search_results = self.searcher.search(query, top_k)
        
        if not search_results:
            return {"answer": "I couldn't find relevant information to answer your question.", "sources": []}
        
        # Step 2: Prepare context from retrieved documents
        print("📝 Step 2: Preparing context...")
        context_parts = []
        sources = []
        
        for i, result in enumerate(search_results, 1):
            context_parts.append(f"[Document {i}] {result['text']}")
            sources.append({
                "document_id": result["metadata"]["document_id"],
                "similarity_score": result["similarity_score"],
                "text_preview": result["text"][:100] + "..."
            })
        
        context = "\n\n".join(context_parts)
        
        # Step 3: Generate answer with LLM
        print("🧠 Step 3: Generating answer with GPT-4o...")
        
        system_prompt = """You are a helpful assistant that answers questions about Azure OpenAI based on provided context.

Instructions:
- Use only the information provided in the context to answer questions
- If the context doesn't contain enough information, say so clearly
- Provide specific, actionable advice when possible
- Reference the document numbers [Document 1], [Document 2], etc. when citing information
- Be concise but comprehensive"""

        user_prompt = f"""Context from knowledge base:
{context}

Question: {query}

Please provide a comprehensive answer based on the context above."""

        try:
            response = self.client.chat.completions.create(
                model=self.gpt_deployment,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=1000,
                temperature=0.7
            )
            
            answer = response.choices[0].message.content
            
            print("✅ Step 4: Answer generated successfully!")
            
            return {
                "query": query,
                "answer": answer,
                "sources": sources,
                "context_used": context,
                "token_usage": response.usage.total_tokens if hasattr(response, 'usage') else None
            }
            
        except Exception as e:
            print(f"❌ Error generating answer: {e}")
            return {"answer": f"Error generating answer: {e}", "sources": sources}
    
    def interactive_rag(self, queries: List[str]) -> List[Dict]:
        """Run RAG on multiple queries"""
        
        print("🤖 INTERACTIVE RAG SYSTEM")
        print("=" * 60)
        
        results = []
        
        for i, query in enumerate(queries, 1):
            print(f"\n{'#'*20} QUESTION {i} {'#'*20}")
            result = self.generate_answer(query)
            results.append(result)
            
            # Display result
            print(f"\n📋 ANSWER:")
            print(f"{result['answer']}")
            
            if result.get('sources'):
                print(f"\n📚 SOURCES USED:")
                for j, source in enumerate(result['sources'], 1):
                    print(f"   {j}. {source['document_id']} (similarity: {source['similarity_score']:.3f})")
            
            print("\n" + "="*60)
        
        return results

# Create complete RAG system
rag_system = RAGSystem(searcher, client, gpt_deployment)

# Test with sample questions
rag_queries = [
    "How can I improve the performance of my Azure OpenAI application?",
    "What security measures should I implement for Azure OpenAI?",
    "I'm getting rate limited. What should I do?",
    "How can I reduce my Azure OpenAI costs?"
]

rag_results = rag_system.interactive_rag(rag_queries)

🤖 INTERACTIVE RAG SYSTEM

#################### QUESTION 1 ####################
🤖 RAG SYSTEM: Answering 'How can I improve the performance of my Azure OpenAI application?'
🔍 Step 1: Retrieving relevant documents...
🔍 Searching for: 'How can I improve the performance of my Azure OpenAI application?'
--------------------------------------------------
📊 Found 3 results:

   🏆 Result 1 (similarity: 0.6610)
   📄 Document: azure_openai_performance
   📝 Text: Azure OpenAI Performance Best Practices: Use connection pooling to reduce latency. Implement exponen...

   🏆 Result 2 (similarity: 0.5955)
   📄 Document: azure_openai_troubleshooting
   📝 Text: Azure OpenAI Troubleshooting Guide: Check regional proximity to deployment for latency issues. Revie...

   🏆 Result 3 (similarity: 0.5454)
   📄 Document: azure_openai_cost
   📝 Text: Azure OpenAI Cost Management: Monitor token usage with Azure Monitor. Set up billing alerts and quot...
📝 Step 2: Preparing context...
🧠 Step 3: Generating answer wi

## 📊 Step 7: Analyze RAG System Performance

Let's analyze how well our RAG system is performing and what we can improve.

In [9]:
def analyze_rag_performance(rag_results: List[Dict], search_results: Dict[str, List[Dict]]):
    """Analyze the performance of our RAG system"""
    
    print("📊 RAG SYSTEM PERFORMANCE ANALYSIS")
    print("=" * 60)
    
    # 1. Retrieval Quality Analysis
    print("\n🔍 RETRIEVAL QUALITY ANALYSIS")
    print("-" * 40)
    
    retrieval_stats = {
        "total_queries": len(search_results),
        "avg_top_similarity": 0,
        "min_similarity": 1.0,
        "max_similarity": 0,
        "queries_with_high_confidence": 0  # similarity > 0.8
    }
    
    for query, results in search_results.items():
        if results:
            top_similarity = results[0]["similarity_score"]
            retrieval_stats["avg_top_similarity"] += top_similarity
            retrieval_stats["min_similarity"] = min(retrieval_stats["min_similarity"], top_similarity)
            retrieval_stats["max_similarity"] = max(retrieval_stats["max_similarity"], top_similarity)
            
            if top_similarity > 0.8:
                retrieval_stats["queries_with_high_confidence"] += 1
    
    retrieval_stats["avg_top_similarity"] /= retrieval_stats["total_queries"]
    
    print("📈 Retrieval Statistics:")
    print(f"   Average top similarity: {retrieval_stats['avg_top_similarity']:.4f}")
    print(f"   Min similarity: {retrieval_stats['min_similarity']:.4f}")
    print(f"   Max similarity: {retrieval_stats['max_similarity']:.4f}")
    print(f"   High confidence queries: {retrieval_stats['queries_with_high_confidence']}/{retrieval_stats['total_queries']}")
    
    # 2. Generation Quality Analysis
    print("\n🧠 GENERATION QUALITY ANALYSIS")
    print("-" * 40)
    
    generation_stats = {
        "successful_generations": 0,
        "avg_answer_length": 0,
        "sources_used_distribution": {},
        "total_tokens": 0
    }
    
    for result in rag_results:
        if result.get("answer") and not result["answer"].startswith("Error"):
            generation_stats["successful_generations"] += 1
            generation_stats["avg_answer_length"] += len(result["answer"])
            
            # Count sources used
            num_sources = len(result.get("sources", []))
            generation_stats["sources_used_distribution"][num_sources] = generation_stats["sources_used_distribution"].get(num_sources, 0) + 1
            
            if result.get("token_usage"):
                generation_stats["total_tokens"] += result["token_usage"]
    
    if generation_stats["successful_generations"] > 0:
        generation_stats["avg_answer_length"] /= generation_stats["successful_generations"]
    
    print("📈 Generation Statistics:")
    print(f"   Successful generations: {generation_stats['successful_generations']}/{len(rag_results)}")
    print(f"   Average answer length: {generation_stats['avg_answer_length']:.0f} characters")
    print(f"   Total tokens used: {generation_stats['total_tokens']}")
    print(f"   Sources used distribution: {generation_stats['sources_used_distribution']}")
    
    # 3. Coverage Analysis
    print("\n📚 KNOWLEDGE BASE COVERAGE")
    print("-" * 40)
    
    documents_referenced = set()
    for result in rag_results:
        for source in result.get("sources", []):
            documents_referenced.add(source["document_id"])
    
    total_documents = len(documents)
    coverage_percentage = (len(documents_referenced) / total_documents) * 100
    
    print(f"   Documents in knowledge base: {total_documents}")
    print(f"   Documents referenced: {len(documents_referenced)}")
    print(f"   Coverage percentage: {coverage_percentage:.1f}%")
    print(f"   Referenced documents: {', '.join(documents_referenced)}")
    
    # 4. Recommendations
    print("\n💡 OPTIMIZATION RECOMMENDATIONS")
    print("-" * 40)
    
    recommendations = []
    
    if retrieval_stats["avg_top_similarity"] < 0.7:
        recommendations.append("📊 Consider improving chunking strategy - low similarity scores suggest poor retrieval")
    
    if retrieval_stats["queries_with_high_confidence"] < len(search_results) * 0.8:
        recommendations.append("🔧 Add more diverse content to knowledge base for better coverage")
    
    if generation_stats["avg_answer_length"] < 200:
        recommendations.append("📝 Answers might be too brief - consider adjusting generation prompts")
    
    if coverage_percentage < 80:
        recommendations.append("📚 Some knowledge base documents are not being used - review content relevance")
    
    if not recommendations:
        recommendations.append("✅ RAG system performance looks good!")
    
    for rec in recommendations:
        print(f"   {rec}")
    
    return {
        "retrieval_stats": retrieval_stats,
        "generation_stats": generation_stats,
        "coverage_percentage": coverage_percentage
    }

# Analyze the RAG system performance
performance_analysis = analyze_rag_performance(rag_results, search_results)

📊 RAG SYSTEM PERFORMANCE ANALYSIS

🔍 RETRIEVAL QUALITY ANALYSIS
----------------------------------------
📈 Retrieval Statistics:
   Average top similarity: 0.5054
   Min similarity: 0.4240
   Max similarity: 0.6618
   High confidence queries: 0/5

🧠 GENERATION QUALITY ANALYSIS
----------------------------------------
📈 Generation Statistics:
   Successful generations: 4/4
   Average answer length: 1650 characters
   Total tokens used: 2381
   Sources used distribution: {3: 4}

📚 KNOWLEDGE BASE COVERAGE
----------------------------------------
   Documents in knowledge base: 4
   Documents referenced: 4
   Coverage percentage: 100.0%
   Referenced documents: azure_openai_performance, azure_openai_security, azure_openai_cost, azure_openai_troubleshooting

💡 OPTIMIZATION RECOMMENDATIONS
----------------------------------------
   📊 Consider improving chunking strategy - low similarity scores suggest poor retrieval
   🔧 Add more diverse content to knowledge base for better coverage


## 🔧 Step 8: RAG System Improvements

Let's implement some improvements to our RAG system based on the analysis.

In [10]:
class ImprovedRAGSystem(RAGSystem):
    """Enhanced RAG system with better retrieval and generation"""
    
    def __init__(self, searcher: SemanticSearcher, client: AzureOpenAI, gpt_deployment: str):
        super().__init__(searcher, client, gpt_deployment)
        self.query_cache = {}  # Cache for repeated queries
        self.feedback_scores = {}  # Simple feedback tracking
    
    def enhanced_retrieval(self, query: str, top_k: int = 5, diversity_threshold: float = 0.8) -> List[Dict]:
        """Enhanced retrieval with diversity filtering"""
        
        # Get more results initially
        initial_results = self.searcher.search(query, top_k=top_k*2)
        
        if not initial_results:
            return []
        
        # Apply diversity filtering to avoid redundant results
        diverse_results = [initial_results[0]]  # Always include top result
        
        for result in initial_results[1:]:
            is_diverse = True
            
            # Check if this result is too similar to already selected ones
            for selected in diverse_results:
                if result["similarity_score"] > diversity_threshold and \
                   result["metadata"]["document_id"] == selected["metadata"]["document_id"]:
                    is_diverse = False
                    break
            
            if is_diverse and len(diverse_results) < top_k:
                diverse_results.append(result)
        
        print(f"🔍 Enhanced retrieval: {len(diverse_results)} diverse results from {len(initial_results)} candidates")
        return diverse_results
    
    def generate_enhanced_answer(self, query: str, use_cache: bool = True) -> Dict:
        """Generate answer with enhanced context and formatting"""
        
        # Check cache first
        if use_cache and query in self.query_cache:
            print("💾 Using cached result")
            return self.query_cache[query]
        
        print(f"🚀 ENHANCED RAG: Answering '{query}'")
        print("=" * 60)
        
        # Enhanced retrieval
        search_results = self.enhanced_retrieval(query, top_k=4)
        
        if not search_results:
            return {"answer": "I couldn't find relevant information to answer your question.", "sources": []}
        
        # Enhanced context preparation
        context_parts = []
        sources = []
        
        for i, result in enumerate(search_results, 1):
            # Add document context with metadata
            doc_info = f"Document {i} (from {result['metadata']['document_id']}, relevance: {result['similarity_score']:.3f})"
            context_parts.append(f"[{doc_info}]\n{result['text']}")
            
            sources.append({
                "document_id": result["metadata"]["document_id"],
                "similarity_score": result["similarity_score"],
                "text_preview": result["text"][:150] + "...",
                "chunk_id": result["metadata"]["chunk_id"]
            })
        
        context = "\n\n".join(context_parts)
        
        # Enhanced system prompt
        enhanced_system_prompt = """You are an expert Azure OpenAI consultant that provides detailed, actionable advice.

Instructions:
- Use ONLY the provided context to answer questions
- Provide specific, step-by-step guidance when possible  
- Include relevant code examples or configuration details if mentioned in context
- Structure your response with clear headings and bullet points
- Always cite your sources using [Document X] references
- If context is insufficient, clearly state what additional information would be needed
- Focus on practical implementation details"""

        user_prompt = f"""Knowledge Base Context:
{context}

User Question: {query}

Please provide a comprehensive, well-structured answer with specific actionable steps. Use the document references to support your recommendations."""

        try:
            response = self.client.chat.completions.create(
                model=self.gpt_deployment,
                messages=[
                    {"role": "system", "content": enhanced_system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=1500,  # Increased for more comprehensive answers
                temperature=0.3   # Lower temperature for more focused responses
            )
            
            answer = response.choices[0].message.content
            
            result = {
                "query": query,
                "answer": answer,
                "sources": sources,
                "context_used": context,
                "token_usage": response.usage.total_tokens if hasattr(response, 'usage') else None,
                "retrieval_method": "enhanced",
                "num_documents_searched": len(search_results)
            }
            
            # Cache the result
            if use_cache:
                self.query_cache[query] = result
            
            return result
            
        except Exception as e:
            print(f"❌ Error generating enhanced answer: {e}")
            return {"answer": f"Error generating answer: {e}", "sources": sources}
    
    def compare_systems(self, queries: List[str]) -> Dict:
        """Compare basic vs enhanced RAG performance"""
        
        print("⚖️ COMPARING BASIC vs ENHANCED RAG")
        print("=" * 60)
        
        comparison_results = {
            "basic": [],
            "enhanced": []
        }
        
        for query in queries:
            print(f"\n🔄 Testing query: '{query}'")
            
            # Basic RAG
            basic_result = super().generate_answer(query)
            comparison_results["basic"].append(basic_result)
            
            # Enhanced RAG  
            enhanced_result = self.generate_enhanced_answer(query)
            comparison_results["enhanced"].append(enhanced_result)
            
            # Quick comparison
            basic_length = len(basic_result.get("answer", ""))
            enhanced_length = len(enhanced_result.get("answer", ""))
            
            print(f"   📊 Basic answer: {basic_length} chars")
            print(f"   📊 Enhanced answer: {enhanced_length} chars")
            print(f"   📈 Improvement: {((enhanced_length - basic_length) / basic_length * 100):.1f}% longer" if basic_length > 0 else "")
        
        return comparison_results

# Create enhanced RAG system
enhanced_rag = ImprovedRAGSystem(searcher, client, gpt_deployment)

# Test enhanced system
enhanced_queries = [
    "How can I optimize Azure OpenAI performance for a production application?",
    "What's the complete security checklist for Azure OpenAI deployment?"
]

print("🚀 TESTING ENHANCED RAG SYSTEM")
print("=" * 60)

for query in enhanced_queries:
    print(f"\n{'='*20} ENHANCED RAG TEST {'='*20}")
    result = enhanced_rag.generate_enhanced_answer(query)
    
    print(f"\n📋 ENHANCED ANSWER:")
    print(result["answer"])
    
    print(f"\n📊 METADATA:")
    print(f"   🔍 Documents searched: {result.get('num_documents_searched', 'N/A')}")
    print(f"   🎯 Tokens used: {result.get('token_usage', 'N/A')}")
    print(f"   📚 Sources: {len(result.get('sources', []))}")

🚀 TESTING ENHANCED RAG SYSTEM

==================== ENHANCED RAG TEST ====================
🚀 ENHANCED RAG: Answering 'How can I optimize Azure OpenAI performance for a production application?'
🔍 Searching for: 'How can I optimize Azure OpenAI performance for a production application?'
--------------------------------------------------
📊 Found 8 results:

   🏆 Result 1 (similarity: 0.6971)
   📄 Document: azure_openai_performance
   📝 Text: Azure OpenAI Performance Best Practices: Use connection pooling to reduce latency. Implement exponen...

   🏆 Result 2 (similarity: 0.6181)
   📄 Document: azure_openai_troubleshooting
   📝 Text: Azure OpenAI Troubleshooting Guide: Check regional proximity to deployment for latency issues. Revie...

   🏆 Result 3 (similarity: 0.6079)
   📄 Document: azure_openai_security
   📝 Text: Azure OpenAI Security Best Practices: Use Azure AD authentication and managed identities. Implement ...

   🏆 Result 4 (similarity: 0.6047)
   📄 Document: azure_openai_cost
 

## 🎯 Step 9: Understanding the RAG Pipeline

Let's visualize and understand what we've built and how it compares to production systems.

In [11]:
def visualize_rag_pipeline():
    """Visualize the complete RAG pipeline we've implemented"""
    
    print("🔄 COMPLETE RAG PIPELINE VISUALIZATION")
    print("=" * 60)
    
    pipeline_steps = [
        ("📄 Document Ingestion", "Raw text documents → Document chunks"),
        ("🔪 Chunking Strategy", "Split documents by sentences/tokens/semantics"),
        ("🧠 Embedding Generation", "Text chunks → 3072-dimensional vectors"),
        ("🗃️ Vector Storage", "Store embeddings + metadata in searchable index"),
        ("🔍 Query Processing", "User question → Query embedding"),
        ("📊 Similarity Search", "Find top-k most similar document chunks"),
        ("📝 Context Assembly", "Combine retrieved chunks into context"),
        ("🤖 Answer Generation", "LLM generates answer using retrieved context"),
        ("📚 Citation & Sources", "Include references to source documents")
    ]
    
    print("🔄 Pipeline Steps:")
    for i, (step_name, description) in enumerate(pipeline_steps, 1):
        print(f"   {i}. {step_name}")
        print(f"      → {description}")
    
    print(f"\n📊 Our Implementation Statistics:")
    print(f"   📄 Documents processed: {len(documents)}")
    print(f"   🔪 Total chunks created: {len(embedded_chunks)}")
    print(f"   🧠 Embeddings generated: {len([c for c in embedded_chunks if c.get('embedding')])}")
    print(f"   🗃️ Vector store size: {len(vector_store.vectors)}")
    print(f"   📐 Embedding dimensions: {len(vector_store.vectors[0]) if vector_store.vectors else 0}")
    
    return pipeline_steps

def compare_with_production_systems():
    """Compare our simple implementation with production RAG systems"""
    
    print("\n⚖️ PRODUCTION RAG SYSTEMS COMPARISON")
    print("=" * 60)
    
    comparison_table = [
        ("Component", "Our Implementation", "Production Systems"),
        ("Vector Database", "In-memory NumPy arrays", "Pinecone, Weaviate, FAISS, Azure AI Search"),
        ("Chunking", "Simple sentence splitting", "Advanced semantic chunking, overlap strategies"),
        ("Embeddings", "Azure OpenAI text-embedding-3-large", "Multiple embedding models, fine-tuned embeddings"),
        ("Search Algorithm", "Cosine similarity", "Hybrid search, keyword + semantic, filters"),
        ("Retrieval Strategy", "Top-k similarity", "MMR, diversity ranking, re-ranking models"),
        ("Context Assembly", "Simple concatenation", "Smart truncation, importance weighting"),
        ("Generation", "Single GPT-4o call", "Multi-step reasoning, chain-of-thought"),
        ("Caching", "Simple query cache", "Multi-level caching, embedding cache"),
        ("Monitoring", "Basic statistics", "A/B testing, relevance scoring, user feedback"),
        ("Scaling", "Single machine", "Distributed systems, load balancing"),
        ("Updates", "Manual re-indexing", "Incremental updates, versioning")
    ]
    
    print("📊 Feature Comparison:")
    for component, our_impl, production in comparison_table:
        print(f"\n🔧 {component}:")
        print(f"   📝 Our approach: {our_impl}")
        print(f"   🏭 Production: {production}")
    
    print(f"\n💡 Key Production Enhancements:")
    print(f"   🔄 Real-time indexing with incremental updates")
    print(f"   🎯 Multi-modal search (text + images + metadata)")
    print(f"   📊 Advanced ranking algorithms (BM25 + semantic)")
    print(f"   🔍 Query understanding and expansion")
    print(f"   📈 A/B testing for retrieval strategies")
    print(f"   🛡️ Security and access control")
    print(f"   📱 API rate limiting and scaling")
    print(f"   📋 Comprehensive logging and analytics")

# Visualize our pipeline
pipeline_info = visualize_rag_pipeline()

# Compare with production
compare_with_production_systems()

🔄 COMPLETE RAG PIPELINE VISUALIZATION
🔄 Pipeline Steps:
   1. 📄 Document Ingestion
      → Raw text documents → Document chunks
   2. 🔪 Chunking Strategy
      → Split documents by sentences/tokens/semantics
   3. 🧠 Embedding Generation
      → Text chunks → 3072-dimensional vectors
   4. 🗃️ Vector Storage
      → Store embeddings + metadata in searchable index
   5. 🔍 Query Processing
      → User question → Query embedding
   6. 📊 Similarity Search
      → Find top-k most similar document chunks
   7. 📝 Context Assembly
      → Combine retrieved chunks into context
   8. 🤖 Answer Generation
      → LLM generates answer using retrieved context
   9. 📚 Citation & Sources
      → Include references to source documents

📊 Our Implementation Statistics:
   📄 Documents processed: 4
   🔪 Total chunks created: 9
   🧠 Embeddings generated: 9
   🗃️ Vector store size: 9
   📐 Embedding dimensions: 3072

⚖️ PRODUCTION RAG SYSTEMS COMPARISON
📊 Feature Comparison:

🔧 Component:
   📝 Our approach: O

## 🎓 What You've Accomplished

✅ **Complete RAG Pipeline**: Built a full retrieval-augmented generation system from scratch  
✅ **Document Chunking**: Implemented multiple chunking strategies for different use cases  
✅ **Vector Embeddings**: Generated semantic embeddings using Azure OpenAI  
✅ **Vector Storage**: Created a searchable vector database with metadata  
✅ **Semantic Search**: Implemented cosine similarity search for relevant documents  
✅ **Answer Generation**: Combined retrieval with LLM generation for comprehensive answers  
✅ **Performance Analysis**: Analyzed and optimized RAG system performance  
✅ **Production Insights**: Understood the gap between demo and production RAG systems  

## 🧠 Key Learnings About RAG

### 🔍 **The RAG Process is NOT Magic**
- It's a straightforward pipeline: chunk → embed → store → search → generate
- Each step has trade-offs and optimization opportunities
- The quality depends heavily on your chunking strategy and knowledge base

### 📊 **Embeddings are Compression, Not Encryption**
- Embeddings capture semantic meaning in vectors
- Similar meaning = similar vectors (high cosine similarity)
- You can't reverse embeddings back to original text
- Multiple texts can have similar embeddings

### 🎯 **Retrieval Quality Determines Answer Quality**
- Bad chunking = bad retrieval = bad answers
- Similarity scores help evaluate retrieval confidence
- Diversity in retrieved documents improves answer comprehensiveness

### 🏭 **Production RAG is Complex**
- Our demo shows the concepts, but production systems are much more sophisticated
- Real systems use hybrid search, re-ranking, caching, and continuous optimization
- Monitoring and evaluation are crucial for maintaining quality

## 🚀 Next Steps for Production RAG

### 📈 **Immediate Improvements**
1. **Better Chunking**: Use semantic chunking or overlapping windows
2. **Hybrid Search**: Combine keyword search (BM25) with semantic search
3. **Re-ranking**: Use cross-encoder models to re-rank retrieved results
4. **Query Expansion**: Expand user queries for better retrieval

### 🏗️ **Production Architecture**
1. **Vector Database**: Use Pinecone, Weaviate, or Azure AI Search
2. **Embedding Pipeline**: Batch processing with error handling and retries
3. **API Layer**: REST API with authentication, rate limiting, and caching
4. **Monitoring**: Track retrieval quality, answer relevance, and user satisfaction

### 🔧 **Advanced Features**
1. **Multi-modal RAG**: Support for images, tables, and structured data
2. **Conversational RAG**: Maintain context across multiple turns
3. **Domain Adaptation**: Fine-tune embeddings for specific domains
4. **Evaluation Framework**: Automated testing of retrieval and generation quality

## 💡 Understanding Production RAG Systems

**Now when you see enterprise RAG systems like:**
- **Azure AI Search** with vector search capabilities
- **OpenAI Assistants** with Vector Stores (like in notebook 05)
- **LangChain** with vector store integrations
- **Microsoft Copilot** with enterprise data

**You understand that under the hood, they're all doing the same fundamental process we just implemented:**
1. Chunk documents into searchable pieces
2. Convert chunks to embeddings for semantic similarity
3. Store embeddings with metadata for fast retrieval
4. Search for relevant chunks using vector similarity
5. Use retrieved chunks as context for LLM generation

The difference is in the sophistication of each step and the infrastructure to support enterprise-scale deployments! 🎉